# QLoRA Fine-tuning — Llama 3.2 3B + MedAlpaca

Treino no Kaggle T4×2. Adapter publicado no HuggingFace Hub.

**Setup:** Adicionar `HF_TOKEN` como Kaggle Secret antes de rodar.

| Parâmetro | Valor |
|-----------|-------|
| Modelo base | meta-llama/Llama-3.2-3B-Instruct |
| Dataset | medalpaca/medical_meadow_medqa |
| Amostras de treino | 8.000 |
| LoRA rank | r=16, α=32 |
| Quantização | 4-bit NF4 (QLoRA) |
| Épocas | 2 |


In [ ]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets rouge-score huggingface_hub

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
login(token=hf_token)
print('HF login OK')

In [ ]:
import torch
print(f'GPUs disponíveis: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
MODEL_ID        = 'meta-llama/Llama-3.2-3B-Instruct'
HUB_MODEL_ID    = 'LucasSGarrido/llama-3.2-3b-medqa'
DATASET_ID      = 'medalpaca/medical_meadow_medqa'
MAX_TRAIN       = 8000
MAX_VAL         = 500
MAX_SEQ_LENGTH  = 512   # reduzido de 1024 — evita OOM no T4
BATCH_SIZE      = 2     # reduzido de 4
GRAD_ACC        = 8     # aumentado de 4 — effective batch = 16
EPOCHS          = 2
LR              = 2e-4
SEED            = 42
OUTPUT_DIR      = '/kaggle/working/checkpoints'

SYSTEM_PROMPT = (
    'You are a helpful medical assistant. '
    'Answer questions accurately based on medical knowledge.'
)

In [ ]:
from datasets import load_dataset

print('Carregando MedAlpaca...')
full_ds = load_dataset(DATASET_ID, split='train')
full_ds = full_ds.shuffle(seed=SEED).select(range(MAX_TRAIN + MAX_VAL))
print(f'Total: {len(full_ds)} exemplos | Colunas: {full_ds.column_names}')
print('Exemplo:', full_ds[0])

split = full_ds.train_test_split(test_size=MAX_VAL, seed=SEED)
train_ds = split['train']
val_ds   = split['test']
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
def format_instruction(example):
    instruction = example.get('instruction', '')
    input_text  = example.get('input', '')
    output      = example.get('output', '')
    user_content = instruction
    if input_text.strip():
        user_content = f'{instruction}\n\n{input_text}'
    text = (
        '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n'
        f'{SYSTEM_PROMPT}<|eot_id|>'
        '<|start_header_id|>user<|end_header_id|>\n\n'
        f'{user_content}<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
        f'{output}<|eot_id|>'
    )
    return {'text': text}

train_ds = train_ds.map(format_instruction, remove_columns=train_ds.column_names)
val_ds   = val_ds.map(format_instruction,   remove_columns=val_ds.column_names)
print('Formatação OK. Preview:')
print(train_ds[0]['text'][:400])

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

print('Carregando modelo com 4-bit quantization...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token        = tokenizer.eos_token
tokenizer.padding_side     = 'right'
tokenizer.model_max_length = MAX_SEQ_LENGTH  # força truncação — evita OOM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
)
model.config.use_cache = False
print('Modelo carregado OK')

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)

# T4 não suporta BFloat16 com fp16 AMP — forçar LoRA params para float16
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float16)

model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    warmup_steps=50,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    logging_steps=50,
    save_steps=200,
    eval_steps=200,
    eval_strategy='steps',
    save_strategy='steps',
    load_best_model_at_end=True,
    dataset_text_field='text',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
)

print('Iniciando treino...')
trainer.train()
print('Treino concluído!')

In [ ]:
import random
from rouge_score import rouge_scorer

ASSISTANT_HEADER = '<|start_header_id|>assistant<|end_header_id|>\n\n'

model.eval()
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
random.seed(SEED)
indices = random.sample(range(len(val_ds)), min(100, len(val_ds)))

scores = []
for idx in indices:
    full_text = val_ds[idx]['text']
    ref_start = full_text.rfind(ASSISTANT_HEADER) + len(ASSISTANT_HEADER)
    reference = full_text[ref_start:].replace('<|eot_id|>', '').strip()
    prompt    = full_text[:ref_start]

    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    prediction = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    scores.append(scorer.score(reference, prediction)['rougeL'].fmeasure)

mean_rouge_l = sum(scores) / len(scores)
print(f'ROUGE-L médio (n={len(scores)}): {mean_rouge_l:.4f}')

In [ ]:
print(f'Salvando e publicando em: https://huggingface.co/{HUB_MODEL_ID}')
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
trainer.model.push_to_hub(HUB_MODEL_ID)
tokenizer.push_to_hub(HUB_MODEL_ID)
print('Publicado com sucesso!')

In [ ]:
test_q = 'What is the first-line treatment for hypertension in adults?'
prompt = (
    '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n'
    f'{SYSTEM_PROMPT}<|eot_id|>'
    '<|start_header_id|>user<|end_header_id|>\n\n'
    f'{test_q}<|eot_id|>'
    '<|start_header_id|>assistant<|end_header_id|>\n\n'
)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
new_tokens = out[0][inputs['input_ids'].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
print(f'Q: {test_q}')
print(f'A: {answer}')